In [ ]:
#cost_sensitivity

In [ ]:
"""
Transaction-cost sensitivity analysis.

Why this exists
---------------
The headline backtest used a single 20 bps round-trip cost assumption. But for
this strategy the net result sits close to break-even, so the *sign* of the
conclusion depends on that assumption. Rather than defend one number, we sweep a
range of cost levels and report each method's Sharpe as a function of cost, plus
the break-even cost at which Sharpe crosses zero. This turns an arbitrary
parameter into a quantified statement about how cost-fragile the edge is.

The cost is applied per unit of turnover (sum of absolute daily position
changes), so it already accounts for both legs of every pair and for the extra
turnover induced by 6-monthly re-selection.

Cost levels and the real-world scenarios they represent
-------------------------------------------------------
All values are bps (1 bp = 0.01%) charged on turnover, i.e. blended round-trip
proxies for fee + spread + slippage on the assets actually being traded.

  5 bps  -> Best case. Maker-only execution on a top-tier venue (Binance/
            Binance.US ~0.075-0.10% taker, less as maker / with fee discounts),
            traded ONLY in the most liquid names (BTC, ETH, SOL, XRP, BNB) where
            bid-ask spread and slippage are negligible. Realistic only for a
            disciplined trader posting passive orders in deep books.

 10 bps  -> Realistic large-cap case. Standard taker fees (~0.10%) on liquid
            majors, with minimal slippage. A fair estimate for a retail trader
            transacting the top ~10-15 coins by volume without posting passive.

 20 bps  -> Base case (used in the headline run). Blended fee + modest slippage
            across a MIXED universe that includes mid-caps. Once the basket
            reaches down to names like ATOM, ALGO, GRT, the bid-ask spread and
            market impact start to matter, lifting the effective cost above the
            pure-fee floor. This is the most honest single number for the 46-coin
            universe actually traded here.

 30 bps  -> Conservative / small-cap-inclusive case. The universe contains
            lower-liquidity names (VTHO, DGB, LOOM, LTO, SYS) where spreads can
            be tens of bps and any non-trivial order size moves the book. Also a
            stand-in for trading on a pricier retail venue (e.g. Coinbase
            standard tier, where taker fees alone can reach 0.4-0.6%).

 50 bps  -> Stress case. Dominated by slippage in illiquid alts under size, or a
            high-fee venue. Less a prediction than a robustness floor: if the
            edge survives here it is real; if it dies here, the strategy is only
            viable with tight execution in liquid names.

Read the output as: the strategy is viable only if true all-in cost on the
traded basket is below the break-even bps reported per method.
"""

from __future__ import annotations

import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "src"))

import numpy as np
import pandas as pd

from data_loader import load_panel
from pair_selection import SELECTORS
from backtest import (
    generate_signals, build_positions, backtest, sharpe_ratio,
)
import run_pipeline as rp

# Cost grid in bps; see module docstring for the scenario each one represents.
COST_LEVELS_BPS = [5, 10, 20, 30, 50]


def break_even_cost(prices, positions, oos_start, lo=0.0, hi=200.0, tol=0.1) -> float:
    """
    Bisection search for the round-trip cost (bps) at which OOS Sharpe = 0.
    Returns NaN if Sharpe is already <= 0 at zero cost (no edge to erode).
    """
    s_at_zero = sharpe_ratio(backtest(prices, positions, tcost_bps=0.0).loc[oos_start:])
    if s_at_zero <= 0:
        return float("nan")  # no positive gross edge; never crosses from + to -

    for _ in range(40):
        mid = (lo + hi) / 2
        s = sharpe_ratio(backtest(prices, positions, tcost_bps=mid).loc[oos_start:])
        if abs(s) < 1e-4 or (hi - lo) < tol:
            return mid
        if s > 0:
            lo = mid  # still profitable -> cost can go higher
        else:
            hi = mid
    return (lo + hi) / 2


prices = load_panel(rp.PANEL)
    oos_start = prices.index[0] + pd.DateOffset(days=rp.LOOKBACK_DAYS + 30)
    print(f"Panel: {prices.shape[0]} dates x {prices.shape[1]} coins\n")

    # Run the (expensive) walk-forward ONCE per method, cache positions, then
    # the cost sweep is just cheap re-costing of the same positions.
    positions = {}
    for name, selector in SELECTORS.items():
        print(f"  walk-forward: {name} ...")
        positions[name] = rp.walk_forward(prices, selector, oos_start)

    # Sharpe at each cost level.
    sharpe_table = {}
    for name, pos in positions.items():
        sharpe_table[name] = {
            f"{c}bps": round(sharpe_ratio(backtest(prices, pos, tcost_bps=c).loc[oos_start:]), 3)
            for c in COST_LEVELS_BPS
        }
    sharpe_df = pd.DataFrame(sharpe_table).T

    # Gross (zero-cost) Sharpe and break-even cost per method.
    extra = {}
    for name, pos in positions.items():
        gross = round(sharpe_ratio(backtest(prices, pos, tcost_bps=0.0).loc[oos_start:]), 3)
        be = break_even_cost(prices, pos, oos_start)
        extra[name] = {"Gross(0bps)": gross,
                       "BreakEven_bps": round(be, 1) if be == be else float("nan")}
    extra_df = pd.DataFrame(extra).T

    out = pd.concat([extra_df["Gross(0bps)"], sharpe_df, extra_df["BreakEven_bps"]], axis=1)

    print("\n=== Sharpe vs transaction cost (OOS, by method) ===")
    print(out.to_string())
    print("\nGross(0bps): Sharpe with no costs (raw signal edge).")
    print("BreakEven_bps: round-trip cost at which Sharpe hits 0; NaN = no positive")
    print("gross edge to begin with (strategy unprofitable even free).")

    os.makedirs("results", exist_ok=True)
    out.to_csv("results/cost_sensitivity.csv")
    print("\nSaved -> results/cost_sensitivity.csv")


if __name__ == "__main__":
    main()
